In [1]:
import pandas as pd
from sqlalchemy import create_engine
import urllib.parse
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:5432/{DB_NAME}"
)

In [10]:
df_final = pd.read_sql(
    "SELECT * FROM customer_behavior_final",
    engine
)

print("Data loaded ✅")
df_final.shape


Data loaded ✅


(1000000, 65)

In [9]:
segment_distribution = (
    df_final["customer_segment"]
    .value_counts(normalize=True) * 100
)

segment_distribution


customer_segment
at_risk_customers      54.5613
loyal_customers        36.3878
potential_loyalists     9.0509
Name: proportion, dtype: float64

In [11]:
df_final.groupby("customer_segment")["impulse_buying_score"].mean()


customer_segment
at_risk_customers      2.463112
loyal_customers        8.550841
potential_loyalists    6.000000
Name: impulse_buying_score, dtype: float64

In [14]:
df_final["recommendation_strategy"].value_counts(normalize=True) * 100


recommendation_strategy
Discount and retention offers      54.5613
Premium product recommendations    36.3878
Personalized bundle offers          9.0509
Name: proportion, dtype: float64

In [18]:
segment_summary = df_final.groupby("customer_segment").agg({
    "impulse_buying_score": "mean",
    "return_rate": "mean",
    "premium_subscription": "mean"
})

segment_summary


,impulse_buying_score,return_rate,premium_subscription
customer_segment,,,
at_risk_customers,2.463112,50.020181,0.359931
loyal_customers,8.550841,50.022194,0.358884
potential_loyalists,6.000000,49.843795,0.358439


In [19]:
segment_summary.to_sql(
    "final_business_insights",
    engine,
    if_exists="replace"
)

print("Business insights table saved ✅")



Business insights table saved ✅


segment_summary = df_final.groupby("customer_segment").agg({
    "engagement_score": "mean",
    "impulse_buying_score": "mean",
    "return_rate": "mean",
    "premium_subscription": "mean"
})

segment_summary


In [20]:
recommendation_map = {
    "loyal_customers": "Premium and early-access product recommendations",
    "potential_loyalists": "Personalized bundles and targeted cross-sell offers",
    "at_risk_customers": "Retention discounts and re-engagement campaigns"
}

df_final["recommendation_strategy"] = df_final["customer_segment"].map(recommendation_map)

In [21]:
# Create recommendation mapping from segment to top category
segment_recommendation = {
    "loyal_customers": "premium_electronics",
    "potential_loyalists": "bundled_home_products",
    "at_risk_customers": "discounted_best_sellers"
}

In [22]:
df_final.groupby("customer_segment")["recommendation_strategy"].value_counts()

customer_segment     recommendation_strategy                            
at_risk_customers    Retention discounts and re-engagement campaigns        545613
loyal_customers      Premium and early-access product recommendations       363878
potential_loyalists  Personalized bundles and targeted cross-sell offers     90509
Name: count, dtype: int64

In [23]:
df_final.groupby("customer_segment")["recommendation_strategy"].value_counts(normalize=True) * 100

customer_segment     recommendation_strategy                            
at_risk_customers    Retention discounts and re-engagement campaigns        100.0
loyal_customers      Premium and early-access product recommendations       100.0
potential_loyalists  Personalized bundles and targeted cross-sell offers    100.0
Name: proportion, dtype: float64

# 📊 Step 4: Customer Segment Performance Analysis

#Segment Comparison
1. Impulse Buying Score
- Loyal Customers: 8.55  
- Potential Loyalists: 6.00  
- At Risk Customers: 2.46  

Loyal Customers demonstrate the highest impulse buying behavior, confirming strong purchasing tendencies. At Risk Customers show significantly lower impulsive activity.

2. Return Rate
Return rates are relatively similar across segments, indicating that customer risk classification is driven more by purchasing intensity than by product dissatisfaction.


3. Premium Subscription Rate
Premium subscription adoption remains consistent across segments, suggesting opportunity for targeted upselling among Potential Loyalists.

4. Strategic Insight
The behavioral segmentation effectively distinguishes high-value customers from lower-engagement groups. Loyal Customers represent strong revenue potential and should receive premium product strategies. Potential Loyalists require personalized engagement to increase purchasing intensity. At Risk Customers need retention-focused campaigns to prevent churn and reactivate engagement.